In [14]:

import sys

import numpy as np
import pandas as pd
import torch

sys.path.append("..")
from src.pytorch_models import build_model, select_device
from src.train_utils import load_artifact

device = select_device()
print("Using device:", device)

Using device: cpu


Carregar artefacto e modelo


In [15]:
artifact_path = "modelo_B_mlp_tfidf.pt"
print("Loading artifact:", artifact_path)
artifact = load_artifact(artifact_path, device=device)

model = build_model(artifact["model_name"], artifact["model_config"]).to(device)
model.load_state_dict(artifact["state_dict"])
model.eval()

idx_to_label = artifact["idx_to_label"]

Loading artifact: modelo_B_mlp_tfidf.pt


In [16]:
df_subm = pd.read_csv("../data/subm1.csv", sep=";")

X = artifact["tfidf"].transform(df_subm["Text"].values).astype(np.float32)
x_tensor = torch.tensor(X, dtype=torch.float32, device=device)

In [18]:
with torch.no_grad():
    outputs = model(x_tensor)
    pred_idx = outputs.argmax(dim=1)

pred_idx = pred_idx.cpu().numpy().tolist()
pred_labels = [idx_to_label[int(i)] for i in pred_idx]

df_out = df_subm.copy()
df_out["Labels"] = pred_labels

In [19]:
output_path = "subm1-g1-MEI-B.csv"
df_out.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(df_out))
print("Label counts:")
print(df_out["Labels"].value_counts())

Saved: subm1-g1-MEI-B.csv
Rows: 150
Label counts:
Labels
Google       61
Anthropic    36
OpenAI       21
Meta         17
Human        15
Name: count, dtype: int64
